# Part 1 — Acquisition & Storage: Data Acquisition and Source Verification

**BFT65B — Big Data Analytics in Finance | Student ID: 11230945**

Acquire every Financials quarter from 1984Q1 to 2026Q2, together with the complete Failures and Institutions reference tables. Preserve source fields and verify data completeness, schema consistency and artifact integrity to establish the Bronze layer. No modelling labels or economic corrections are performed at this stage.

## 1. Repository and pinned runtime
Run on a CPU runtime. The package installation and research code use separate Python processes.

In [2]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/NguyenDuong0508/11230945.git"
REPO = Path("/content/11230945")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    actual = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO, text=True).strip()
    if actual != REPO_URL:
        raise RuntimeError(f"Unexpected repository: {actual}")
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)
# The research commands run in fresh Python processes to avoid stale imports.
def command(*args):
    subprocess.run([sys.executable, str(REPO / "run_part1.py"), *map(str, args)], cwd=REPO, check=True)


Git revision: e26eea6d7a3f439c5b1affd07385bd8af14d8313


## 2. Mandatory offline validation
The tests cover source contracts, pagination, checkpoint recovery, CSV schema restoration, Parquet equality and partition pruning. Synthetic fixtures do not call the live API. Continue only after every test passes without skips.

In [3]:

from pathlib import Path
import os
import sys
import subprocess
import xml.etree.ElementTree as ET

TESTS_PASSED = False

print("Python version:", sys.version)
print("Repository:", REPO)

result_path = Path("/content/part1_test_results.xml")
result_path.unlink(missing_ok=True)

env = dict(
    os.environ,
    PYTEST_DISABLE_PLUGIN_AUTOLOAD="1"
)

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "-q",
        "-ra",
        "--tb=short",
        "--show-capture=no",
        f"--junitxml={result_path}",
        "tests"
    ],
    cwd=REPO,
    env=env,
    capture_output=True,
    text=True,
    check=False
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Pytest exit code:", result.returncode)

if not result_path.is_file():
    raise RuntimeError(
        "Pytest did not generate a test report."
    )

root = ET.parse(result_path).getroot()
cases = list(root.iter("testcase"))

failed = []
skipped = []

for case in cases:
    for tag in ("failure", "error", "skipped"):
        detail = case.find(tag)

        if detail is not None:
            name = (
                f"{case.get('classname', '')}"
                f"::{case.get('name', '')}"
            )

            print(f"\n{tag.upper()}: {name}")
            print(detail.get("message", ""))

            if tag != "skipped":
                print(detail.text or "")
                failed.append(name)
            else:
                skipped.append(name)

if result.returncode != 0 or not cases or failed or skipped:
    raise RuntimeError(
        f"Validation failed: "
        f"{len(cases)} tests, "
        f"{len(failed)} failures/errors, "
        f"{len(skipped)} skips. "
        "See diagnostics above."
    )

TESTS_PASSED = True

print(
    f"Mandatory offline validation: PASS "
    f"({len(cases)} tests, zero skips)"
)


Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Repository: /content/11230945
........................................................................ [ 50%]
......................................................................   [100%]
------------- generated xml file: /content/part1_test_results.xml --------------
142 passed in 317.44s (0:05:17)

Pytest exit code: 0
Mandatory offline validation: PASS (142 tests, zero skips)


## 3. Persistent workspace
Keep the source data in Google Drive, outside GitHub. Use this workspace for all subsequent parts.

In [4]:
from google.colab import drive
import json

drive.mount("/content/drive")
WORKSPACE = Path("/content/drive/MyDrive/11230945")
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Workspace:", WORKSPACE)


Mounted at /content/drive
Workspace: /content/drive/MyDrive/11230945


## 4. First-quarter acquisition check
The initial command acquires at most one new financial quarter. It saves a checkpoint, not a complete source. If the workspace already has a sealed source, the command verifies it instead. This is also a live API field and transport check for the first quarter.

In [5]:
assert TESTS_PASSED, "Run the full offline test suite first"
command("acquire", "--workspace", WORKSPACE, "--max-new-quarters", 1)


## 5. Full acquisition
Resume the pending source and retrieve all remaining quarters plus Failures and Institutions without outcome filters. Every completed artifact is committed to the manifest. If the runtime disconnects, rerun setup, tests, mount and this cell; completed valid checkpoints are reused. Do not start simultaneous acquisitions against the workspace.

In [6]:
assert TESTS_PASSED
command("acquire", "--workspace", WORKSPACE)


## 6. Independent verification
Reopen the sealed artifacts and validate the calendar, API totals, physical files, keys, schema and checksums. The active pointer refers to the exact verified manifest.

In [7]:
assert TESTS_PASSED
command("verify", "--workspace", WORKSPACE)
assert not (WORKSPACE / "pending_source.json").exists()
source_pointer = json.loads((WORKSPACE / "active_source.json").read_text())
SOURCE = WORKSPACE / source_pointer["path"]
source_report = json.loads((SOURCE / "verification.json").read_text())
assert source_report["status"] == "PASS" and source_report["quarter_count"] == 170
assert source_report["manifest_sha256"] == source_pointer["manifest_sha256"]
print("Acquisition accepted:", SOURCE.name)
print("Rows:", source_report["row_counts"])
print("Distinct financial CERTs:", source_report["distinct_financial_certs"])
print("Missing failure CERTs:", source_report["failure_null_cert"])
print("RESTYPE distribution (not model labels):", source_report["failure_restype_counts"])
print("Manifest SHA-256:", source_pointer["manifest_sha256"])


Acquisition accepted: 2026-11-25T173655_344198Z_e6d1bc49
Rows: {'failures': 4117, 'financials': 1682739, 'institutions': 27838}
Distinct financial CERTs: 24064
Missing failure CERTs: 488
RESTYPE distribution (not model labels): [{'rows': 593, 'source_value': 'ASSISTANCE'}, {'rows': 3524, 'source_value': 'FAILURE'}]
Manifest SHA-256: e376152baac058e4dd4fff4582e9afa3c58820fff2f278115a82bff7fd4b3f6e


## 7. Quarterly coverage evidence
Compare retrieved rows with the API total for every quarter, not only the combined row count. Display a compact coverage summary.

In [8]:
from IPython.display import display, Markdown
import html

quarter_stats = source_report["quarter_statistics"]
assert all(q["rows"] == q["api_total"] for q in quarter_stats)
# Small textual summary; no bank-level source rows are displayed.
first, last = quarter_stats[0], quarter_stats[-1]
for q in (first, last):
    print(q["quarter"], f"{q['rows']:,} rows", f"{q['bytes']/2**20:.2f} MiB")
print("Quarters with matching API totals:", len(quarter_stats))
print("Financial partition size (MiB):", round(sum(q["bytes"] for q in quarter_stats)/2**20, 3))


1984-03-31 17,930 rows 2.06 MiB
2026-06-30 4,314 rows 0.74 MiB
Quarters with matching API totals: 170
Financial partition size (MiB): 243.433


## 8. Handoff
Notebook 02 resolves `active_source.json`; do not copy a run ID. Part 2 will use the same immutable source and event fields. Save this executed notebook with its summary outputs, but do not commit the workspace or source data.